# 04 — Drift analysis

Generate a PSI drift snapshot from two temporal slices of the dataset. Writes `mlruns/drift_snapshot.json` consumed by the Streamlit drift-monitor page.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.loader import DataLoader
from src.data.splits import temporal_train_val_test_split
from src.features.pipelines import build_engineered_frame
from src.monitoring.drift import generate_drift_snapshot

sns.set_theme(style='whitegrid', palette='deep')

In [ ]:
loader = DataLoader()
frame = loader.load_sample(n=300_000, random_state=42)
bundle = build_engineered_frame(frame)
split = temporal_train_val_test_split(bundle.frame)
print(f'Reference window: {split.train.shape[0]:,} rows ending before {split.val_start}')
print(f'Target window:    {split.test.shape[0]:,} rows starting at {split.test_start}')

## 1. Generate drift snapshot

PSI is computed per feature against the training-window distribution. Severity bands defined in `src.monitoring.alerts`.

In [ ]:
feature_subset = list(bundle.numerical_columns)[:30]
snapshot = generate_drift_snapshot(
    reference=split.train,
    target=split.test,
    feature_columns=feature_subset,
    reference_window_label=str(split.val_start),
    target_window_label=str(split.test_start),
)
print(f'Features evaluated: {len(snapshot["feature_drift"])}')
print(f'Snapshot written to mlruns/drift_snapshot.json')

## 2. Top drifted features

In [ ]:
drift_frame = pd.DataFrame(snapshot['feature_drift']).sort_values('psi', ascending=True)
severity_colours = {'monitor': '#22c55e', 'warning': '#f59e0b', 'regulator-relevant': '#ef4444'}
colours = drift_frame['severity'].map(severity_colours)
fig, ax = plt.subplots(figsize=(9, max(4, 0.25 * len(drift_frame))))
ax.barh(drift_frame['feature'], drift_frame['psi'], color=colours)
ax.axvline(0.10, color='#f59e0b', linestyle='--', linewidth=1)
ax.axvline(0.25, color='#ef4444', linestyle='--', linewidth=1)
ax.set_xlabel('Population Stability Index')
ax.set_title('PSI by feature (train → test windows)')
plt.tight_layout()
plt.show()
drift_frame[['feature', 'psi', 'severity']]